# Scribal Identification: Clustering & Visualization

Unsupervised clustering of deed embeddings from a ViT-Small backbone pretrained
on the Antwerp charter corpus. The pipeline:

1. **Load** pre-extracted VLAD embeddings (38 400-d per document)
2. **PCA** → 50 dimensions (variance retention, denoising)
3. **UMAP** → 2 dimensions (neighbourhood-preserving projection)
4. **HDBSCAN** → density-based clusters with soft membership probabilities
5. **Visualize** with Bokeh (interactive scatter + Voronoi overlay)
6. **Gallery** of the most central documents per cluster

## 1 — Configuration

In [1]:
import numpy as np
import base64, re, json, os, io
from pathlib import Path
from PIL import Image

from sklearn.decomposition import PCA
from sklearn.cluster import HDBSCAN
from scipy.spatial.distance import cdist, pdist, squareform
from scipy.spatial import Delaunay, Voronoi

from bokeh.plotting import figure, output_file, save
from bokeh.models import ColumnDataSource, HoverTool, Label
from bokeh.palettes import Category20

In [2]:
# ── Paths & data ────────────────────────────────────────────────────
EMBEDDINGS_FILE = "../assets/embeddings/embeddings-fland+antw.npz"
IMAGES_DIR      = "../images/cropped/"
THUMBNAIL_SIZE  = (1000, 1000)
THUMBNAIL_CACHE = "thumbnail_cache.json"

# ── Dimensionality reduction ────────────────────────────────────────
UMAP_N_NEIGHBORS = 15        # controls local vs global structure (try 10–50)
UMAP_MIN_DIST    = 0.1       # tightness of clusters in 2-D (lower = tighter)

# ── HDBSCAN ─────────────────────────────────────────────────────────
HDBSCAN_MIN_CLUSTER_SIZE = 10
HDBSCAN_MIN_SAMPLES      = 5
HDBSCAN_METHOD            = "leaf"   # "leaf" = fine-grained; "eom" = coarser
HDBSCAN_EPSILON           = 0.0      # >0 prevents merging subclusters beyond this dist

# ── Misc ────────────────────────────────────────────────────────────
DISTANCE_METRIC = "euclidean"
RANDOM_SEED     = 42

os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## 2 — Load embeddings & apply exclusions

Each document is represented by a single 38 400-d VLAD vector (100 clusters × 384-d
ViT-Small descriptors), power- and L2-normalised. Documents listed in `exclude.txt`
(e.g. heavily damaged deeds) are dropped before any analysis.

In [3]:
data       = np.load(EMBEDDINGS_FILE)
embeddings = data["embeddings"]
filenames  = data["filenames"]

#exclude = set(f.strip() for f in  ["Genois-1327a", "Genois-1327b", "Genois-1327c", "RA-800r"])
exclude = ["RA-800r"]
keep    = np.array([not any(e in str(fn) for e in exclude) for fn in filenames])

embeddings = embeddings[keep]
filenames  = filenames[keep]
N = len(embeddings)

print(f"Excluded {(~keep).sum()} docs  →  {N} embeddings retained")

Excluded 1 docs  →  1397 embeddings retained


## 3 — Map embeddings to on-disk image files

Embedding filenames use a `0-<ID>_…` scheme; the raw images may or may not
carry the `0-` prefix. We normalise both to a bare numeric ID for matching.

In [4]:
import unicodedata

images_dir   = Path(IMAGES_DIR)
actual_files = [f for f in images_dir.iterdir() if f.is_file()]


def _nfc(s: str) -> str:
    """Normalise to NFC so macOS NFD filenames match NFC names in the .npz."""
    return unicodedata.normalize("NFC", s)


def _extract_id(name: str, prefix: str = r"(?:0-)?") -> str | None:
    """Return the bare numeric ID from a numeric-scheme filename, else None."""
    m = re.match(rf"^{prefix}(\d+)", name)
    return (m.group(1).lstrip("0") or "0") if m else None


# Two indices: by NFC stem (works for ANY name, incl. Büdingen1r) and by
# numeric ID (fallback for the legacy 0-<ID>_… scheme).
disk_by_stem = {_nfc(f.stem): f for f in actual_files}
disk_by_id   = {_extract_id(f.name): f for f in actual_files if _extract_id(f.name)}


def find_disk(fn) -> "Path | None":
    """Locate the on-disk image for an embedding filename.
    Stem match (NFC-normalised) first; numeric-ID scheme only as a fallback."""
    stem = _nfc(Path(str(fn)).stem)
    if stem in disk_by_stem:
        return disk_by_stem[stem]
    eid = _extract_id(str(fn))
    if eid and eid in disk_by_id:
        return disk_by_id[eid]
    return None


print(f"{len(actual_files)} images on disk  ->  "
      f"{len(disk_by_stem)} by stem, {len(disk_by_id)} by numeric ID")


1398 images on disk  ->  1398 by stem, 1392 by numeric ID


In [5]:
import torch

def kRNN(X, k, r=True):
    X = torch.tensor(X)
    S = torch.mm(X, X.t())
    # initial ranking list
    _, initial_rank = S.topk(k=S.shape[0], dim=-1, largest=True, sorted=True)
    kNN = initial_rank[:, 1:k+1]

    reranked = torch.zeros(X.shape)
    for i in range(X.shape[0]):
        feat = X[i]
        nn = kNN[i]

        if r:
            rnn = [X[j] for j in nn if i in kNN[j]]
        else:
            rnn = [X[j] for j in nn]

        if len(rnn) > 0:
            rnn = torch.concat(rnn).view(-1, rnn[0].shape[0])
            reranked[i] = (feat + torch.sum(rnn, dim=0)) / (rnn.shape[0] + 1)
        else:
            reranked[i] = feat

    reranked_norm = torch.norm(reranked, p=2, dim=1, keepdim=True)
    reranked = reranked.div(reranked_norm.expand_as(reranked))  
    return reranked

In [6]:
KRNN_K = 15  # neighbourhood size, try 5, 10, 20

#print(f"Applying kRNN re-ranking (k={KRNN_K}, reciprocal=True)...")
#embeddings = kRNN(embeddings, k=KRNN_K, r=True).numpy()
#print(f"  Input:  {embeddings.shape}")
#print(f"  Output: {embeddings.shape}")

## 4 — PCA (38 400 → 50)

Aggressive reduction before UMAP. 50 components is a practical sweet spot
that retains the dominant variance while cutting noise and speeding up the
downstream distance computation.

In [7]:
N_PCA = 150
pca = PCA(n_components=N_PCA, whiten=True, random_state=RANDOM_SEED)
#pca = PCA(n_components=N_PCA, random_state=RANDOM_SEED)
embeddings_pca = pca.fit_transform(embeddings)

print(f"Explained variance (by PCs): {pca.explained_variance_ratio_.sum():.2%}")

Explained variance (by PCs): 61.71%


## 5 — Pairwise distance matrix

Computed on the 50-d PCA space. The full N × N matrix is passed to UMAP as a
precomputed distance matrix, which gives us explicit control over the metric.

In [8]:
D_emb = squareform(pdist(embeddings_pca, metric=DISTANCE_METRIC))
print(f"Distance matrix: {D_emb.shape}  ({DISTANCE_METRIC})")

Distance matrix: (1397, 1397)  (euclidean)


## 6 — UMAP (50 → 2)

Non-linear projection that preserves local neighbourhood structure.
We feed the precomputed distance matrix so the UMAP metric matches our PCA metric.

In [9]:
from umap import UMAP as UMAPReducer

reducer = UMAPReducer(
    n_components=2,
    n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,
    metric="precomputed",
    random_state=RANDOM_SEED,
)
embeddings_2d = reducer.fit_transform(D_emb)
print(f"UMAP output: {embeddings_2d.shape}")

/Users/mikekestemont/miniconda3/envs/bayes/lib/python3.10/site-packages/umap/umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")
/Users/mikekestemont/miniconda3/envs/bayes/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


UMAP output: (1397, 2)


## 7 — HDBSCAN parameter sweep

Quick grid search over `min_cluster_size` (mcs) and `min_samples` (ms) to inspect
the stability of the cluster count. Each cell shows `<clusters>c / <noise%>`.

In [10]:
mcs_vals = [4, 5, 6, 8, 10, 12, 15, 20]
ms_vals  = [3, 5, 7, 10]

rows = []
for mcs in mcs_vals:
    row = {"mcs": mcs}
    for ms in ms_vals:
        labs = HDBSCAN(
            min_cluster_size=mcs, min_samples=ms,
            metric="euclidean", cluster_selection_method="eom",
            cluster_selection_epsilon=HDBSCAN_EPSILON,
        ).fit_predict(embeddings_2d)
        nc = labs.max() + 1
        nn = (labs == -1).sum()
        row[f"ms={ms}"] = f"{nc}c / {nn/len(labs)*100:.1f}%"
    rows.append(row)

# Pretty-print as a simple table
header = f"{'mcs':>5}" + "".join(f"  {k:>12}" for k in rows[0] if k != "mcs")
print(header)
print("-" * len(header))
for r in rows:
    line = f"{r['mcs']:>5}"
    for k, v in r.items():
        if k != "mcs":
            line += f"  {v:>12}"
    print(line)

  mcs          ms=3          ms=5          ms=7         ms=10
-------------------------------------------------------------
    4  128c / 15.8%   83c / 25.3%   59c / 31.6%   30c / 23.1%
    5   92c / 12.9%   73c / 26.3%   58c / 34.1%   31c / 24.5%
    6   85c / 12.6%   68c / 27.3%   55c / 31.9%   30c / 24.0%
    8   70c / 16.8%   58c / 29.1%   52c / 32.7%   27c / 21.4%
   10   56c / 19.1%   48c / 32.7%   32c / 20.2%   27c / 21.4%
   12   44c / 22.6%   22c / 10.1%   27c / 20.8%    12c / 0.6%
   15    13c / 1.9%    13c / 1.7%    13c / 1.1%    11c / 0.5%
   20    10c / 3.6%    10c / 3.8%    10c / 3.1%     9c / 2.9%


## 8 — HDBSCAN clustering (final)

Run with the chosen parameters on the 2-D UMAP embedding. `leaf` mode tends
to produce more fine-grained subclusters — appropriate here because adjacent
subclusters may represent the *same* scribe at different career stages and we
prefer to validate merges through the human annotation campaign rather than
commit to them automatically.

In [11]:
eps_label = f", eps={HDBSCAN_EPSILON}" if HDBSCAN_EPSILON > 0 else ""
print(f"HDBSCAN: mcs={HDBSCAN_MIN_CLUSTER_SIZE}, ms={HDBSCAN_MIN_SAMPLES}, "
      f"method={HDBSCAN_METHOD}{eps_label}")

clusterer = HDBSCAN(
    min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
    min_samples=HDBSCAN_MIN_SAMPLES,
    metric="euclidean",
    cluster_selection_method=HDBSCAN_METHOD,
    cluster_selection_epsilon=HDBSCAN_EPSILON,
    store_centers="medoid",
)
hdb_labels = clusterer.fit_predict(embeddings_2d)
hdb_probs  = clusterer.probabilities_

n_clusters = hdb_labels.max() + 1
n_noise    = (hdb_labels == -1).sum()
print(f"→ {n_clusters} clusters  +  {n_noise} noise ({n_noise/N:.1%})")
print()

for ci in range(n_clusters):
    print(f"  H{ci:>2}: {(hdb_labels == ci).sum():>3} docs")

HDBSCAN: mcs=10, ms=5, method=leaf
→ 50 clusters  +  473 noise (33.9%)

  H 0:  21 docs
  H 1:  24 docs
  H 2:  18 docs
  H 3:  26 docs
  H 4:  17 docs
  H 5:  29 docs
  H 6:  13 docs
  H 7:  16 docs
  H 8:  16 docs
  H 9:  25 docs
  H10:  14 docs
  H11:  26 docs
  H12:  10 docs
  H13:  13 docs
  H14:  13 docs
  H15:  25 docs
  H16:  18 docs
  H17:  10 docs
  H18:  14 docs
  H19:  12 docs
  H20:  15 docs
  H21:  13 docs
  H22:  10 docs
  H23:  26 docs
  H24:  12 docs
  H25:  20 docs
  H26:  10 docs
  H27:  30 docs
  H28:  21 docs
  H29:  19 docs
  H30:  11 docs
  H31:  21 docs
  H32:  12 docs
  H33:  10 docs
  H34:  13 docs
  H35:  13 docs
  H36:  11 docs
  H37:  14 docs
  H38:  21 docs
  H39:  42 docs
  H40:  13 docs
  H41:  71 docs
  H42:  12 docs
  H43:  11 docs
  H44:  25 docs
  H45:  10 docs
  H46:  16 docs
  H47:  13 docs
  H48:  20 docs
  H49:  29 docs


## 9 — Medoid identification

For each cluster, select the single document closest to the cluster centroid
among those with near-maximal membership probability. These medoids serve as
the visual "prototypes" for each cluster in the paper.

In [12]:
medoid_indices = []
for ci in range(n_clusters):
    members  = np.where(hdb_labels == ci)[0]
    probs    = hdb_probs[members]
    # restrict to top-probability members (within 0.01 of the max)
    top      = members[probs >= probs.max() - 0.01]
    centroid = embeddings_2d[members].mean(axis=0, keepdims=True)
    dists    = cdist(embeddings_2d[top], centroid).flatten()
    medoid   = top[np.argmin(dists)]
    medoid_indices.append(medoid)
    print(f"  H{ci}: {filenames[medoid]}  (prob {hdb_probs[medoid]:.3f})")

  H0: 733o.png  (prob 1.000)
  H1: 1135o.png  (prob 1.000)
  H2: 527o.png  (prob 1.000)
  H3: 856o.png  (prob 1.000)
  H4: 286o.png  (prob 0.993)
  H5: 357o.png  (prob 1.000)
  H6: 573o.png  (prob 1.000)
  H7: 825o.png  (prob 1.000)
  H8: 871o.png  (prob 1.000)
  H9: 1003o.png  (prob 1.000)
  H10: 430o.png  (prob 1.000)
  H11: 1328o.png  (prob 1.000)
  H12: 973o.png  (prob 1.000)
  H13: 638o.png  (prob 1.000)
  H14: 1391o.png  (prob 1.000)
  H15: 539o.png  (prob 1.000)
  H16: 232o.png  (prob 1.000)
  H17: 43o.png  (prob 1.000)
  H18: 989o.png  (prob 1.000)
  H19: 958o.png  (prob 1.000)
  H20: 738o.png  (prob 1.000)
  H21: 921o.png  (prob 1.000)
  H22: 604o.png  (prob 1.000)
  H23: 116o.png  (prob 1.000)
  H24: 1362o.png  (prob 1.000)
  H25: 799o.png  (prob 1.000)
  H26: 706o.png  (prob 1.000)
  H27: 1365o.png  (prob 1.000)
  H28: 822o.png  (prob 1.000)
  H29: 858o.png  (prob 1.000)
  H30: 144o.png  (prob 1.000)
  H31: 1019o.png  (prob 1.000)
  H32: 1059o.png  (prob 1.000)
  H33: 647o.p

## 10 — Thumbnail generation (with disk cache)

Base64-encoded PNG thumbnails are embedded in the Bokeh HTML so that
hovering over a point shows the deed image. A JSON cache avoids
re-encoding on repeated runs.

In [13]:
if Path(THUMBNAIL_CACHE).exists():
    with open(THUMBNAIL_CACHE) as f:
        thumb_cache = json.load(f)
else:
    thumb_cache = {}

thumbnails  = []
matched     = 0
cache_hits  = 0
cache_dirty = False
missing     = []

for fn in filenames:
    key = _nfc(Path(str(fn)).stem)          # cache key: stem, works for any name

    if key in thumb_cache:
        thumbnails.append(thumb_cache[key])
        matched += 1; cache_hits += 1; continue

    img_path = find_disk(fn)                 # stem-first, numeric-ID fallback
    if img_path and img_path.exists():
        img = Image.open(img_path)
        img.thumbnail(THUMBNAIL_SIZE)
        if img.mode in ("RGBA", "P"):
            img = img.convert("RGB")
        buf = io.BytesIO()
        img.save(buf, format="PNG")
        b64 = f"data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}"
        thumbnails.append(b64)
        thumb_cache[key] = b64
        cache_dirty = True
        matched += 1
    else:
        thumbnails.append("")
        missing.append(str(fn))

if cache_dirty:
    with open(THUMBNAIL_CACHE, "w") as f:
        json.dump(thumb_cache, f)

print(f"Thumbnails: {matched}/{N} matched  ({cache_hits} cached, "
      f"{matched - cache_hits} new)")
if missing:
    print(f"  {len(missing)} still without a thumbnail, e.g.: {missing[:5]}")


Thumbnails: 1397/1397 matched  (1397 cached, 0 new)


## 11 — Interactive UMAP scatter plot (Bokeh)

Three glyph layers: noise points (grey diamonds), clustered points (coloured
circles sized by membership probability), and medoid stars. A Voronoi diagram
over the cluster centroids is drawn as dashed lines to delineate the regions.

In [14]:
palette = Category20[20]

hdb_colors = ["#888888" if l == -1 else palette[l % 20] for l in hdb_labels]
hdb_sizes  = [10 if l == -1 else 6 + p * 8
              for l, p in zip(hdb_labels, hdb_probs)]

output_filename = "umap_hdbscan.html"
output_file(output_filename)

# Split indices
noise_idx = np.where(hdb_labels == -1)[0]
clust_idx = np.where(hdb_labels != -1)[0]

# ── Data sources ────────────────────────────────────────────────────
source_noise = ColumnDataSource(dict(
    x=embeddings_2d[noise_idx, 0], y=embeddings_2d[noise_idx, 1],
    filename=[str(filenames[i]) for i in noise_idx],
    thumbnail=[thumbnails[i] for i in noise_idx],
    hdb_prob=[f"{hdb_probs[i]:.2f}" for i in noise_idx],
))

source_clustered = ColumnDataSource(dict(
    x=embeddings_2d[clust_idx, 0], y=embeddings_2d[clust_idx, 1],
    filename=[str(filenames[i]) for i in clust_idx],
    thumbnail=[thumbnails[i] for i in clust_idx],
    hdb_cluster=[str(hdb_labels[i]) for i in clust_idx],
    hdb_prob=[f"{hdb_probs[i]:.2f}" for i in clust_idx],
    color=[hdb_colors[i] for i in clust_idx],
    size=[hdb_sizes[i] for i in clust_idx],
))

source_medoids = ColumnDataSource(dict(
    x=embeddings_2d[medoid_indices, 0], y=embeddings_2d[medoid_indices, 1],
    filename=[str(filenames[i]) for i in medoid_indices],
    thumbnail=[thumbnails[i] for i in medoid_indices],
    hdb_cluster=[str(hdb_labels[i]) for i in medoid_indices],
    hdb_prob=[f"{hdb_probs[i]:.2f}" for i in medoid_indices],
    color=[hdb_colors[i] for i in medoid_indices],
    label=[f"H{i}" for i in range(n_clusters)],
))

# ── Figure ──────────────────────────────────────────────────────────
eps_tag = f", eps={HDBSCAN_EPSILON}" if HDBSCAN_EPSILON > 0 else ""
p = figure(
    title=(f"UMAP(n={UMAP_N_NEIGHBORS}, d={UMAP_MIN_DIST}) → "
           f"HDBSCAN(mcs={HDBSCAN_MIN_CLUSTER_SIZE}, ms={HDBSCAN_MIN_SAMPLES}, "
           f"{HDBSCAN_METHOD}{eps_tag}) — {n_clusters} clusters, {n_noise} noise"),
    width=1400, height=1000,
    tools="pan,wheel_zoom,box_zoom,reset,save",
)

### 11a — Voronoi overlay

Dashed lines show the Voronoi diagram of cluster centroids, giving a visual
sense of which region of the UMAP space "belongs" to each cluster. Infinite
ridges are extended outward from the centroid barycentre.

In [15]:
all_centroids = np.array([
    embeddings_2d[hdb_labels == ci].mean(axis=0) for ci in range(n_clusters)
])

if n_clusters >= 3:
    vor = Voronoi(all_centroids)

    # Finite ridges
    for v0, v1 in vor.ridge_vertices:
        if v0 >= 0 and v1 >= 0:
            p.line(
                [vor.vertices[v0, 0], vor.vertices[v1, 0]],
                [vor.vertices[v0, 1], vor.vertices[v1, 1]],
                line_color="#cccccc", line_width=1.5,
                line_dash="dashed", line_alpha=0.9,
            )

    # Infinite ridges — project outward from the centre of mass
    x_span = np.ptp(embeddings_2d[:, 0]) + 2
    y_span = np.ptp(embeddings_2d[:, 1]) + 2
    center = all_centroids.mean(axis=0)

    for pts, (v0, v1) in zip(vor.ridge_points, vor.ridge_vertices):
        if v0 >= 0 and v1 >= 0:
            continue
        finite_v = v0 if v0 >= 0 else v1
        origin   = vor.vertices[finite_v]
        midpoint = all_centroids[pts].mean(axis=0)
        tangent  = all_centroids[pts[1]] - all_centroids[pts[0]]
        normal   = np.array([-tangent[1], tangent[0]])
        normal  /= np.linalg.norm(normal)
        if np.dot(midpoint - center, normal) < 0:
            normal = -normal
        far = origin + normal * max(x_span, y_span) * 2
        p.line([origin[0], far[0]], [origin[1], far[1]],
               line_color="#cccccc", line_width=1.5,
               line_dash="dashed", line_alpha=0.7)

elif n_clusters == 2:
    mid     = all_centroids.mean(axis=0)
    tangent = all_centroids[1] - all_centroids[0]
    normal  = np.array([-tangent[1], tangent[0]])
    normal /= np.linalg.norm(normal)
    span    = max(np.ptp(embeddings_2d[:, 0]), np.ptp(embeddings_2d[:, 1])) * 2
    p.line([mid[0] - normal[0]*span, mid[0] + normal[0]*span],
           [mid[1] - normal[1]*span, mid[1] + normal[1]*span],
           line_color="#cccccc", line_width=1.5,
           line_dash="dashed", line_alpha=0.7)

print(f"Voronoi overlay: {n_clusters} centroids")

Voronoi overlay: 50 centroids


### 11b — Scatter glyphs & hover tooltips

In [16]:
HOVER_HTML = '''<div>
  <div><strong>@filename</strong></div>
  <div>{extra}</div>
  <div><img src="@thumbnail"
       style="max-width:1000px; max-height:1000px;"></div>
</div>'''

# Noise diamonds
r_noise = p.scatter("x", "y", source=source_noise,
                    marker="diamond", size=10, alpha=0.55,
                    color="#888888", line_color="#555555", line_width=1,
                    legend_label="noise")

# Clustered circles
r_clust = p.scatter("x", "y", source=source_clustered,
                    size="size", alpha=0.6, color="color",
                    legend_field="hdb_cluster")

# Medoid stars
r_med = p.scatter("x", "y", source=source_medoids,
                  marker="star", size=25, alpha=1.0, color="color",
                  line_color="black", line_width=2,
                  legend_label="MEDOIDS")

p.add_tools(HoverTool(renderers=[r_noise], tooltips=HOVER_HTML.format(
    extra="<em>noise</em> · prob @hdb_prob")))
p.add_tools(HoverTool(renderers=[r_clust], tooltips=HOVER_HTML.format(
    extra="H@hdb_cluster · prob @hdb_prob")))
p.add_tools(HoverTool(renderers=[r_med],   tooltips=HOVER_HTML.format(
    extra="<strong>MEDOID @label</strong> · H@hdb_cluster · prob @hdb_prob")))


# Cluster labels next to medoids
for i, idx in enumerate(medoid_indices):
    p.add_layout(Label(x=embeddings_2d[idx, 0], y=embeddings_2d[idx, 1],
                       text=f" H{i}", text_font_size="11pt",
                       text_font_style="bold", text_color="black"))

# ── Highlight specific documents of interest ────────────────────────
import unicodedata

HIGHLIGHT_FILES = ["Büdingen1r", "Büdingen1v", "332o", "Genois-1327a", "Genois-1327b", "Genois-1327c", "RA-800r"]

def _norm(s):
    return unicodedata.normalize("NFC", s)

for i in range(N):
    fn_stem = Path(str(filenames[i])).stem
    if _norm(fn_stem) in [_norm(h) for h in HIGHLIGHT_FILES]:
        p.add_layout(Label(
            x=embeddings_2d[i, 0],
            y=embeddings_2d[i, 1],
            text=f"  {fn_stem}",
            text_font_size="16pt",
            text_font_style="bold",
            text_color="#CC0000",
        ))
        p.scatter(
            [embeddings_2d[i, 0]], [embeddings_2d[i, 1]],
            marker="circle", size=18, alpha=0.9,
            color="#CC0000", line_color="black", line_width=2,
        )
        print(f"  ★ Highlighted: {fn_stem} at ({embeddings_2d[i, 0]:.2f}, {embeddings_2d[i, 1]:.2f}), cluster={hdb_labels[i]}")

# ── Annotate all files with a "digit(s)-" prefix ───────────────────
import re

for i in range(N):
    fn_stem = Path(str(filenames[i])).stem
    if re.match(r"^\d+-", fn_stem):
        p.add_layout(Label(
            x=embeddings_2d[i, 0],
            y=embeddings_2d[i, 1],
            text=f"  {fn_stem}",
            text_font_size="14pt",
            text_font_style="bold",
            text_color="#0055AA",
        ))
        p.scatter(
            [embeddings_2d[i, 0]], [embeddings_2d[i, 1]],
            marker="triangle", size=16, alpha=0.9,
            color="#0055AA", line_color="black", line_width=2,
        )
        print(f"  ▲ Annotated: {fn_stem} at ({embeddings_2d[i, 0]:.2f}, {embeddings_2d[i, 1]:.2f}), cluster={hdb_labels[i]}")

p.legend.visible = False
p.xaxis.axis_label = "UMAP 1"
p.yaxis.axis_label = "UMAP 2"
save(p)

p.legend.visible = False
p.xaxis.axis_label = "UMAP 1"
p.yaxis.axis_label = "UMAP 2"
save(p)

print(f"Saved {output_filename} ({Path(output_filename).stat().st_size/1024/1024:.1f} MB)")

  ★ Highlighted: 332o at (4.20, -4.65), cluster=-1
  ★ Highlighted: Büdingen1r at (4.18, -4.61), cluster=-1
  ★ Highlighted: Büdingen1v at (4.12, -4.66), cluster=-1
  ★ Highlighted: Genois-1327a at (4.23, -4.73), cluster=-1
  ★ Highlighted: Genois-1327b at (4.28, -4.73), cluster=-1
  ★ Highlighted: Genois-1327c at (4.23, -4.70), cluster=-1
Saved umap_hdbscan.html (256.6 MB)


In [22]:
# ═══════════════════════════════════════════════════════════════════════
# UMAP scatter coloured by annotated HAND GROUP  (separate HTML file)
# No medoid stars, no noise diamonds — every charter is a coloured circle
# with its hand_group number inside. Only hand groups represented by at
# least two charters are coloured/numbered; singleton groups are demoted
# to the un-grouped (grey) layer. HIGHLIGHT_FILES keep a red ring + label.
# ═══════════════════════════════════════════════════════════════════════
import colorsys
from collections import Counter
import pandas as pd
from bokeh.plotting import figure, save
from bokeh.models import ColumnDataSource, HoverTool, LabelSet
from bokeh.resources import INLINE

META_XLSX     = "../hands-leroy/metadata-matched.xlsx"
HAND_COLUMN   = "hand_group"
KEY_COLUMN    = None          # column identifying the charter; None = auto-detect
OUTPUT_HTML   = "umap_handgroups.html"
CIRCLE_SIZE   = 20            # px; big enough to hold a 1–2 digit number
MIN_GROUP_SIZE = 2            # only plot hand groups with at least this many charters
SHOW_UNMATCHED = True         # faint grey dots for charters with no (kept) hand_group

HIGHLIGHT_FILES = ["Büdingen1r", "Büdingen1v", "332o",
                   "Genois-1327a", "Genois-1327b", "Genois-1327c", "RA-800r"]

# ── Small helpers ───────────────────────────────────────────────────
def _fmt(h):
    return str(int(h)) if isinstance(h, float) and h.is_integer() else str(h)

def _sortkey(v):
    try:    return (0, float(v))
    except (TypeError, ValueError): return (1, str(v))

# ── Load metadata ───────────────────────────────────────────────────
meta = pd.read_excel(META_XLSX)
assert HAND_COLUMN in meta.columns, \
    f"{HAND_COLUMN!r} not found. Columns: {list(meta.columns)}"

file_stems = [_nfc(Path(str(fn)).stem) for fn in filenames]
stem_set   = set(file_stems)
id_to_stem = {}
for s in stem_set:
    eid = _extract_id(s)
    if eid:
        id_to_stem.setdefault(eid, s)

def _resolve_stem(key: str):
    """Map an xlsx key cell to a filename stem: NFC-stem first, numeric-ID fallback."""
    ks = _nfc(Path(str(key)).stem)
    if ks in stem_set:
        return ks
    eid = _extract_id(str(key))
    if eid and eid in id_to_stem:
        return id_to_stem[eid]
    return None

# ── Pick the key column (most rows that resolve to a known charter) ──
if KEY_COLUMN is None:
    scored = [(c, meta[c].dropna().map(lambda v: _resolve_stem(v) is not None).sum())
              for c in meta.columns if c != HAND_COLUMN]
    KEY_COLUMN, best = max(scored, key=lambda t: t[1])
    print(f"Auto-detected key column: {KEY_COLUMN!r}  ({best}/{len(meta)} rows resolve)")

# ── Build  stem -> hand_group  map ──────────────────────────────────
stem_to_hand = {}
for key, hand in zip(meta[KEY_COLUMN], meta[HAND_COLUMN]):
    if pd.isna(hand):
        continue
    s = _resolve_stem(key)
    if s is not None:
        stem_to_hand[s] = hand

hands = [stem_to_hand.get(s) for s in file_stems]

# ── Keep only hand groups with >= MIN_GROUP_SIZE charters ───────────
counts       = Counter(h for h in hands if h is not None)
valid_groups = {g for g, c in counts.items() if c >= MIN_GROUP_SIZE}
dropped      = sorted((g for g, c in counts.items() if c < MIN_GROUP_SIZE), key=_sortkey)
hands        = [h if h in valid_groups else None for h in hands]   # singletons -> grey
if dropped:
    print(f"Dropped {len(dropped)} group(s) with < {MIN_GROUP_SIZE} charters: "
          f"{[_fmt(g) for g in dropped]}")

n_matched = sum(h is not None for h in hands)
print(f"Charters in a kept hand_group: {n_matched} / {N}")

# ── Colours: one distinct hue per kept hand group ───────────────────
groups = sorted(valid_groups, key=_sortkey)

def _palette(n):
    out = []
    for i in range(n):
        r, g, b = colorsys.hsv_to_rgb(i / max(n, 1), 0.62, 0.92)
        out.append(f"#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}")
    return out

group_color = dict(zip(groups, _palette(len(groups))))

def _text_on(hexc):  # black/white label depending on circle luminance
    r, g, b = int(hexc[1:3], 16), int(hexc[3:5], 16), int(hexc[5:7], 16)
    return "#000000" if (0.299*r + 0.587*g + 0.114*b) > 150 else "#ffffff"

print(f"Hand groups plotted: {len(groups)}  ->  {[_fmt(g) for g in groups]}")


grank   = {g: i for i, g in enumerate(groups)}
idx_has = sorted([i for i in range(N) if hands[i] is not None], key=lambda i: grank[hands[i]])
idx_no  = [i for i in range(N) if hands[i] is None]

HOVER_HTML = ('<div><div><strong>@filename</strong></div>'
              '<div>{extra}</div>'
              '<div><img src="@thumbnail" style="max-width:1000px;max-height:1000px;"></div></div>')

p = figure(
    title=(f"UMAP — coloured by {HAND_COLUMN} (≥{MIN_GROUP_SIZE} charters/group)  "
           f"({len(groups)} groups, {n_matched}/{N} charters)"),
    width=1400, height=1000,
    tools="pan,wheel_zoom,box_zoom,reset,save",
)

# faint grey background for un-grouped charters (drawn first, underneath)
if SHOW_UNMATCHED and idx_no:
    src_no = ColumnDataSource(dict(
        x=embeddings_2d[idx_no, 0], y=embeddings_2d[idx_no, 1],
        filename=[str(filenames[i]) for i in idx_no],
        thumbnail=[thumbnails[i] for i in idx_no],
    ))
    r_no = p.scatter("x", "y", source=src_no, size=6, color="#dddddd",
                     line_color="#bbbbbb", alpha=0.5)
    p.add_tools(HoverTool(renderers=[r_no],
                          tooltips=HOVER_HTML.format(extra="<em>no hand group</em>")))

# coloured, numbered charters
src = ColumnDataSource(dict(
    x=embeddings_2d[idx_has, 0], y=embeddings_2d[idx_has, 1],
    filename=[str(filenames[i]) for i in idx_has],
    thumbnail=[thumbnails[i] for i in idx_has],
    hand=[_fmt(hands[i]) for i in idx_has],
    color=[group_color[hands[i]] for i in idx_has],
    tcolor=[_text_on(group_color[hands[i]]) for i in idx_has],
))
r_has = p.scatter("x", "y", source=src, size=CIRCLE_SIZE, color="color",
                  line_color="black", line_width=0.5, alpha=0.95,)
# the hand_group number, centred inside each circle
p.text("x", "y", text="hand", source=src, text_color="tcolor",
       text_font_size="8pt", text_font_style="bold",
       text_align="center", text_baseline="middle")
p.add_tools(HoverTool(renderers=[r_has],
                      tooltips=HOVER_HTML.format(extra=f"{HAND_COLUMN} @hand")))

# ── Special-interest charters: red ring + red label (preserved) ─────
# Hollow ring (not the old solid red dot) so colour/number stay readable.
# Drawn for every HIGHLIGHT file, even if its group was dropped as a singleton.
hl_stems = {_nfc(h) for h in HIGHLIGHT_FILES}
hl_idx   = [i for i in range(N) if _nfc(Path(str(filenames[i])).stem) in hl_stems]

if hl_idx:
    src_hl = ColumnDataSource(dict(
        x=embeddings_2d[hl_idx, 0], y=embeddings_2d[hl_idx, 1],
        label=[f"  {Path(str(filenames[i])).stem}" for i in hl_idx],
    ))
    p.scatter("x", "y", source=src_hl, size=CIRCLE_SIZE + 12, marker="circle",
              fill_alpha=0.0, line_color="#CC0000", line_width=3)
    p.add_layout(LabelSet(x="x", y="y", text="label", source=src_hl,
                          text_color="#CC0000", text_font_size="14pt",
                          text_font_style="bold"))
    for i in hl_idx:
        stem = Path(str(filenames[i])).stem
        h = hands[i]
        print(f"  ★ Highlighted: {stem}  hand_group={_fmt(h) if h is not None else '—'}  "
              f"at ({embeddings_2d[i,0]:.2f}, {embeddings_2d[i,1]:.2f})")
    missing_hl = sorted(hl_stems - {_nfc(Path(str(filenames[i])).stem) for i in hl_idx})
    if missing_hl:
        print(f"  ! HIGHLIGHT_FILES not found among embeddings: {missing_hl}")

p.xaxis.axis_label, p.yaxis.axis_label = "UMAP 1", "UMAP 2"

save(p, filename=OUTPUT_HTML, title=f"UMAP — {HAND_COLUMN}", resources=INLINE)
print(f"Saved {OUTPUT_HTML} ({Path(OUTPUT_HTML).stat().st_size/1024/1024:.1f} MB)")

Auto-detected key column: 'volgnummer'  (2784/2816 rows resolve)
Dropped 22 group(s) with < 2 charters: ['2', '19', '25', '31', '36', '37', '49', '52', '83', '84', '88', '95', '97', '105', '107', '113', '115', '116', '117', '119', '120', '123']
Charters in a kept hand_group: 439 / 1397
Hand groups plotted: 76  ->  ['1', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '20', '21', '22', '23', '24', '26', '27', '29', '32', '33', '34', '35', '38', '39', '41', '42', '43', '44', '45', '46', '50', '53', '54', '55', '56', '57', '58', '60', '61', '63', '65', '66', '68', '69', '71', '75', '76', '78', '80', '82', '85', '86', '87', '89', '90', '91', '93', '94', '96', '99', '100', '101', '102', '103', '104', '106', '108', '111', '112', '129', '130', "108'"]
  ★ Highlighted: 332o  hand_group=—  at (4.20, -4.65)
  ★ Highlighted: Büdingen1r  hand_group=—  at (4.18, -4.61)
  ★ Highlighted: Büdingen1v  hand_group=—  at (4.12, -4.66)
  ★ Highlighted: Genois-1327a  hand_group=—

## Hard query

In [18]:
# ═══════════════════════════════════════════════════════════════════════
# Nearest-neighbour search — rank documents by similarity to a query set
# (drop-in replacement for the exemplar-SVM ranker)
# ═══════════════════════════════════════════════════════════════════════
import unicodedata
from sklearn.preprocessing import normalize

QUERY_FILES = ["Büdingen1r", "Büdingen1v"]

SPACE     = "pca"     # "raw" = 38 400-d VLAD (no PCA loss); "pca" = embeddings_pca
METRIC    = "cosine"  # "cosine" or "euclidean"
AGGREGATE = "max"     # combine multiple queries: "max" (nearest single query)
                      # or "mean" (similarity to the query centroid)


def _norm(s):
    return unicodedata.normalize("NFC", s)


query_stems = [_norm(q) for q in QUERY_FILES]
query_mask  = np.array([_norm(Path(str(fn)).stem) in query_stems for fn in filenames])
print(f"Query documents: {query_mask.sum()} / {N}  ({QUERY_FILES})")
assert query_mask.any(), "No query files matched filenames — check QUERY_FILES / normalisation"

# ── Choose the search space ─────────────────────────────────────────
# Plain NN has no hyperplane to overfit, so the raw VLAD space is safe here
# (unlike the SVM, which needed PCA for a well-posed fit). Switch to "pca"
# to rank in the same space as the UMAP/HDBSCAN clustering.
X = embeddings if SPACE == "raw" else embeddings_pca
Q = X[query_mask]

# ── Similarity of every doc to each query (higher = closer) ─────────
if METRIC == "cosine":
    sims = normalize(X) @ normalize(Q).T          # (N, n_query) in [-1, 1]
elif METRIC == "euclidean":
    sims = -cdist(X, Q, metric="euclidean")       # negate: larger = nearer
else:
    raise ValueError(f"unknown METRIC: {METRIC}")

# ── Collapse the per-query columns into one score per document ──────
scores = sims.max(axis=1) if AGGREGATE == "max" else sims.mean(axis=1)

# ── Rank, excluding the query documents themselves ─────────────────
ranking          = np.argsort(-scores)
ranking_no_query = [i for i in ranking if not query_mask[i]]

print(f"\nTop 20 nearest documents  (space={SPACE}, metric={METRIC}, "
      f"aggregate={AGGREGATE}):\n")
print(f"{'Rank':>4}  {'Score':>8}  {'Cluster':>7}  {'Prob':>5}  Filename")
print("-" * 70)
for rank, idx in enumerate(ranking_no_query[:20], 1):
    fn     = Path(str(filenames[idx])).stem
    cl     = hdb_labels[idx]
    pr     = hdb_probs[idx]
    marker = " ◄" if fn in HIGHLIGHT_FILES else ""
    print(f"{rank:>4}  {scores[idx]:>8.3f}  H{cl:>5}  {pr:.2f}   {fn}{marker}")


Query documents: 2 / 1397  (['Büdingen1r', 'Büdingen1v'])

Top 20 nearest documents  (space=pca, metric=cosine, aggregate=max):

Rank     Score  Cluster   Prob  Filename
----------------------------------------------------------------------
   1     0.587  H   -1  0.00   332o ◄
   2     0.517  H   -1  0.00   Genois-1327c ◄
   3     0.450  H   -1  0.00   Genois-1327b ◄
   4     0.430  H   -1  0.00   Genois-1327a ◄
   5     0.375  H   -1  0.00   650o
   6     0.294  H   21  1.00   560o
   7     0.265  H   -1  0.00   1044o
   8     0.257  H   41  0.85   1358o
   9     0.256  H   -1  0.00   209o
  10     0.254  H   -1  0.00   355o
  11     0.241  H   -1  0.00   198o
  12     0.234  H   27  0.81   114o
  13     0.234  H   -1  0.00   176o
  14     0.232  H   28  0.94   802o
  15     0.221  H   -1  0.00   441o
  16     0.218  H   -1  0.00   93o
  17     0.216  H   35  1.00   659o
  18     0.216  H   -1  0.00   775o
  19     0.205  H   36  1.00   1098o
  20     0.202  H   35  0.86   23o
